In [ ]:
import numpy as np,tensorflow as tf,networkx as nx,matplotlib.pyplot as plt
np.random.seed(14);tf.random.set_seed(14)
# Five-word sentence represented by five 2-dimensional word vectors
words=["Deep","learning","models","learn","sequences"]
tokens=np.array([[.2,.8],[.6,.1],[.9,.4],[.3,.7],[.5,.5]],dtype="float32")
rnn=tf.keras.layers.SimpleRNN(3,return_sequences=True,return_state=True)
inputs=tf.keras.Input((5,2));seq_out,last_state=rnn(inputs);model=tf.keras.Model(inputs,[seq_out,last_state])
_ = model(tokens[None,...]); Wx,Wh,b=rnn.get_weights()
dense=tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(1));_ = dense(seq_out)
Wy,by=dense.layer.get_weights()
states=[];h=np.zeros(3,dtype=np.float32)
for xt in tokens:h=np.tanh(xt@Wx+h@Wh+b);states.append(h.copy())
states=np.array(states);keras_states=model.predict(tokens[None,...],verbose=0)[0][0]
print("Words:",words);print("Input weight shape:",Wx.shape,"Recurrent weight shape:",Wh.shape,"Output weight shape:",Wy.shape)
print("Manual hidden states:\n",np.round(states,4))
print("Maximum difference from Keras:",np.max(np.abs(states-keras_states)))
G=nx.DiGraph();pos={}
for i in range(5):
 G.add_node(f"x{i+1}\n{words[i]}",pos=(i,2));G.add_node(f"h{i+1}",pos=(i,1));G.add_node(f"y{i+1}",pos=(i,0))
 G.add_edge(f"x{i+1}\n{words[i]}",f"h{i+1}",label="W_xh");G.add_edge(f"h{i+1}",f"y{i+1}",label="W_hy")
 if i:G.add_edge(f"h{i}",f"h{i+1}",label="W_hh")
pos=nx.get_node_attributes(G,"pos")
plt.figure(figsize=(13,4));nx.draw(G,pos,with_labels=True,node_size=1500,node_color="lightblue",arrows=True,font_size=8);nx.draw_networkx_edge_labels(G,pos,edge_labels=nx.get_edge_attributes(G,"label"));plt.title("SimpleRNN unfolded over five words");plt.show()
plt.figure(figsize=(8,4));plt.plot(range(1,6),states,marker="o");plt.xticks(range(1,6),words);plt.xlabel("Time step / word");plt.ylabel("Hidden-state activation");plt.title("Hidden-state values across time");plt.grid();plt.legend(["Unit 1","Unit 2","Unit 3"]);plt.show()
print("Output layer weight and bias:",Wy.shape,by.shape)